# Deploy a hosted agent from source code

Deploy a hosted agent to Microsoft Foundry from a zip of Python source: no container image, no registry.

[08-03](../08-03-hosted-agents/08-03-00-hosted-agents.md) built a Docker image, pushed it to ACR and
registered it with `ContainerConfiguration`. This notebook registers the same kind of Agent Framework
agent with `CodeConfiguration` instead: you upload `main.py` and `requirements.txt`, name a runtime
(`python_3_13`) and an entry point, and Foundry builds and runs the sandbox for you.

The agent, `contoso-rm-notes-agent`, keeps meeting notes for Contoso Wealth relationship managers in
its session's `$HOME`. [08-12-02](08-12-02-manage-hosted-agent-sessions.ipynb) uses that state to show
how sessions behave. See [`08-12-00-hosted-agent-sessions.md`](08-12-00-hosted-agent-sessions.md) for
the code-versus-container comparison and the session model.

**What this notebook does:**
1. Reads the Team Alpha spoke settings from the repo `.env`
2. Zips the agent code and registers a version with `dependency_resolution="remote_build"`: the service runs pip from `requirements.txt`
3. Waits for the version to become `active` and grants the agent identity the Foundry User role
4. Sends a test prompt through the agent endpoint
5. Shows content-addressable versioning: the same zip and definition return the existing version
6. Downloads the deployed zip and checks its SHA-256
7. Builds a `bundled` zip with the dependencies pre-installed for `manylinux2014` and registers a second version, timing both provisioning paths
8. Probes the idle timeout bounds the service accepts

The session idle timeout is set per version with `SessionConfiguration`. This lab uses the 120-second
minimum so that 08-12-02 can watch a session go idle and resume.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the
   shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `05-foundry-project-pattern-setup` labs:
   - `ALPHA_FOUNDRY_ACCOUNT` - spoke account name (set by the project spoke deployment)
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` - spoke project endpoint (set by the project spoke deployment)
   - `ALPHA_FOUNDRY_CORE_CONNECTION` - the spoke project's APIM connection, e.g. `core-alpha`
   - `CHAT_MODEL` - chat model deployment name, e.g. `gpt-5.4-mini` (set by the core gateway deployment)
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Foundry Project Manager** (or Owner) on the project to create
   hosted agent versions, and **Owner** or **User Access Administrator** on
   `rg-foundry-spoke-alpha-{suffix}` to grant the agent identity a role in Step 6.
5. **`uv`** on the PATH, for the bundled build in Step 10. Docker is not needed.

## Step 1: Configuration

In [1]:
import ast, hashlib, io, json, os, subprocess, time, zipfile
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Suffix - first 6 chars of SHA-256 of subscription ID (matches the core gateway and project spoke naming)
SUBSCRIPTION_ID = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX          = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
TEAM_NAME       = "alpha"
SPOKE_RG        = f"rg-foundry-spoke-{TEAM_NAME}-{SUFFIX}"

# Existing Spoke Alpha account and project (from the project spoke deployment - already in .env)
ACCOUNT_NAME     = os.environ["ALPHA_FOUNDRY_ACCOUNT"]
PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
CORE_CONNECTION  = os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]  # e.g. "core-alpha": the spoke project's APIM connection
CHAT_MODEL       = os.environ["CHAT_MODEL"]                     # e.g. gpt-5.4-mini

AGENT_NAME = "contoso-rm-notes-agent"
AGENT_DIR  = Path("contoso-rm-notes-agent")   # main.py, requirements.in, requirements.txt
BUILD_DIR  = Path("build")                    # zips and the bundled packages/ folder (gitignored)
RUNTIME    = "python_3_13"                    # the service accepts python_3_13, python_3_14 and dotnet_10
IDLE_TIMEOUT_SECONDS = 120                    # the minimum; 08-12-02 waits for a session to go idle

print(f"Suffix:           {SUFFIX}")
print(f"Resource Group:   {SPOKE_RG}")
print(f"Account:          {ACCOUNT_NAME}")
print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"Agent Name:       {AGENT_NAME}")
print(f"Model:            {CORE_CONNECTION}/{CHAT_MODEL}")
print(f"Runtime:          {RUNTIME}, idle timeout {IDLE_TIMEOUT_SECONDS}s")

Suffix:           c2676f
Resource Group:   rg-foundry-spoke-alpha-c2676f
Account:          aif-spoke-alpha-c2676f
Project Endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent Name:       contoso-rm-notes-agent
Model:            core-alpha/gpt-5.4-mini
Runtime:          python_3_13, idle timeout 120s


## Step 2: The agent code

[`contoso-rm-notes-agent/main.py`](contoso-rm-notes-agent/main.py) is the same shape as the 08-03
agent: a `FoundryChatClient` that resolves `core-alpha/<model>` through the project's APIM
connection, an Agent Framework `Agent`, and `ResponsesHostServer(agent).run()` to serve the
Responses protocol. The difference is four function tools that read and write under `$HOME`:

| Tool | What it does |
|---|---|
| `save_note(title, body)` | Writes `$HOME/notes/<slug>.md` |
| `list_notes()` | Lists the notes in this session's workspace |
| `read_file(path)` | Reads an uploaded file or a saved note, relative to `$HOME` |
| `where_am_i()` | Reports the session ID, host name and process start time of the sandbox |

`$HOME` is what the platform persists per session, so the notes become the visible state in 08-12-02.

`requirements.txt` is compiled from `requirements.in` (the command is in its header) and committed,
so the remote build and the bundled build install the same versions.

In [2]:
tree = ast.parse((AGENT_DIR / "main.py").read_text(encoding="utf-8"))
tools = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)
         and any(getattr(d, "id", None) == "tool" for d in n.decorator_list)]
print(f"Tools the agent exposes to the model: {', '.join(tools)}\n")
for name in ("main.py", "requirements.in", "requirements.txt"):
    print(f"  {name:17s} {(AGENT_DIR / name).stat().st_size:6,d} bytes")

Tools the agent exposes to the model: save_note, list_notes, read_file, where_am_i

  main.py            5,091 bytes
  requirements.in      509 bytes
  requirements.txt  12,034 bytes


## Step 3: Build the zip

The zip must be flat: `main.py` and `requirements.txt` at the root, no wrapper folder. The service
dedups versions by the zip's SHA-256, and a zip records a modification time for every entry, so
a zip built twice from identical files would hash differently and mint a new version. `build_zip`
writes fixed timestamps and sorted entries, which makes the hash depend on the content only.

One detail matters when you write entries by hand: the service reads the Unix mode stored in each
entry's external attributes and ignores entries that are not marked as regular files. An entry
written with mode `0o644` and no file-type bits fails provisioning with "No Python dependency
manifest found at the project root", although `requirements.txt` is in the zip. `0o100644`
(regular file, 0644), which `zipfile.write` sets from the file on disk, works.

In [3]:
def build_zip(out_path: Path, files: dict[str, Path]) -> str:
    """Write a flat zip with fixed timestamps and sorted entries; return its SHA-256 hex digest."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_DEFLATED) as z:
        for arcname in sorted(files):
            info = zipfile.ZipInfo(arcname, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16   # Unix mode: a regular file, 0644
            z.writestr(info, files[arcname].read_bytes())
    return hashlib.sha256(out_path.read_bytes()).hexdigest()

REMOTE_ZIP = BUILD_DIR / "remote-build.zip"
REMOTE_SHA = build_zip(REMOTE_ZIP, {
    "main.py":          AGENT_DIR / "main.py",
    "requirements.txt": AGENT_DIR / "requirements.txt",
})
print(f"{REMOTE_ZIP}: {REMOTE_ZIP.stat().st_size:,} bytes")
print(f"entries: {zipfile.ZipFile(REMOTE_ZIP).namelist()}")
print(f"sha256:  {REMOTE_SHA}")

build/remote-build.zip: 4,300 bytes
entries: ['main.py', 'requirements.txt']
sha256:  526edceedc2bec346b058ee950ff082541920bf6278bd6a05771644898e64a63


## Step 4: Register a version with `remote_build`

`create_version_from_code` uploads the zip as one part of a multipart request and the
`HostedAgentDefinition` as the other. The definition carries:

- `code_configuration`: the runtime, the entry point the sandbox runs, and `dependency_resolution`.
  With `remote_build` the service installs `requirements.txt` while provisioning the version.
- `session_configuration`: the idle timeout for every session created on this version, 120 to
  3600 seconds (the default is 900). Changing it means a new version.
- `cpu` and `memory`: one of the sandbox tiers `0.5`/`1Gi`, `1`/`2Gi` or `2`/`4Gi`.
- `protocol_versions`: Responses `2.0.0`, which the current hosting package requires.
- `environment_variables`: what `main.py` reads, exactly as in 08-03.

The SDK computes the zip's SHA-256 and sends it in the `x-ms-code-zip-sha256` header; the service
stores it as `code_configuration.content_hash`.

In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    CodeConfiguration,
    ProtocolVersionRecord,
    AgentEndpointProtocol,
    SessionConfiguration,
)

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

def label(status) -> str:
    """The SDK returns known statuses as enum members; print their value."""
    return getattr(status, "value", status)

def definition(dependency_resolution: str, idle_timeout_seconds: int = IDLE_TIMEOUT_SECONDS) -> HostedAgentDefinition:
    return HostedAgentDefinition(
        protocol_versions=[ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version="2.0.0")],
        cpu="1", memory="2Gi",
        code_configuration=CodeConfiguration(
            runtime=RUNTIME,
            entry_point=["python", "main.py"],
            dependency_resolution=dependency_resolution,
        ),
        session_configuration=SessionConfiguration(idle_timeout_seconds=idle_timeout_seconds),
        environment_variables={
            "PROJECT_ENDPOINT": PROJECT_ENDPOINT,
            "CHAT_MODEL":       f"{CORE_CONNECTION}/{CHAT_MODEL}",
        },
    )

def register(zip_path: Path, dependency_resolution: str, description: str):
    # The SDK reads the stream's name to label the multipart part, so it must end in .zip.
    with zip_path.open("rb") as code:
        return client.agents.create_version_from_code(
            agent_name=AGENT_NAME,
            definition=definition(dependency_resolution),
            code=code,
            description=description,
        )

remote = register(REMOTE_ZIP, "remote_build", "Notes agent; dependencies installed by the service from requirements.txt")
print(f"Agent registered: {remote.name} v{remote.version}, status={label(remote.status)}")
print(f"content_hash:     {remote.definition.code_configuration.content_hash}")
print(f"matches the zip:  {remote.definition.code_configuration.content_hash == REMOTE_SHA}")

Agent registered: contoso-rm-notes-agent v1, status=creating
content_hash:     526edceedc2bec346b058ee950ff082541920bf6278bd6a05771644898e64a63
matches the zip:  True


## Step 5: Wait for the version to become active

A code-based version starts as `creating` and becomes `active` once the sandbox image is built,
or `failed` with an `error` object (for `remote_build`, its message carries the last pip line).
There is nothing to stream yet: no container runs until the first session.

In [5]:
def wait_active(version: str, poll_seconds: int = 10, timeout_minutes: int = 20) -> float:
    """Poll the version until it is active and return the seconds that took. Raise on failed."""
    t0, last = time.time(), None
    while time.time() - t0 < timeout_minutes * 60:
        v = client.agents.get_version(agent_name=AGENT_NAME, agent_version=version)
        if v.status != last:
            print(f"  {time.time() - t0:5.0f}s  {label(v.status)}")
            last = v.status
        if v.status == "active":
            return time.time() - t0
        if v.status == "failed":
            raise RuntimeError(f"version {version} failed: {v.get('error')}")
        time.sleep(poll_seconds)
    raise TimeoutError(f"version {version} still {last} after {timeout_minutes} minutes")

print(f"Waiting for v{remote.version} (remote build: the service runs pip in the cloud)...")
REMOTE_BUILD_SECONDS = wait_active(remote.version)
print(f"v{remote.version} active after {REMOTE_BUILD_SECONDS:.0f}s")

Waiting for v1 (remote build: the service runs pip in the cloud)...


      0s  creating


     51s  active
v1 active after 51s


## Step 6: Grant the agent identity the Foundry User role

Foundry creates a per-agent managed identity (`instance_identity`) at registration. The sandbox
calls the project's Responses API with it, and that call routes through the `core-alpha` connection
to the gateway, so the identity needs **Foundry User** on the spoke project. There is no image to
pull, so unlike 08-03 no AcrPull is needed. The assignment is idempotent: re-running the notebook
reports it as already granted.

In [6]:
version_details    = client.agents.get_version(agent_name=AGENT_NAME, agent_version=remote.version)
AGENT_PRINCIPAL_ID = version_details.instance_identity.principal_id
print(f"Agent identity: {AGENT_PRINCIPAL_ID}")

project_name = PROJECT_ENDPOINT.rstrip('/').split('/')[-1]
project_id = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{SPOKE_RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}/projects/{project_name}"
)

# "Azure AI User" was renamed to "Foundry User"; assign by stable role-definition ID
FOUNDRY_USER_ROLE_ID = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
r = subprocess.run(
    f"az role assignment create --assignee-object-id {AGENT_PRINCIPAL_ID} "
    f"--assignee-principal-type ServicePrincipal --role {FOUNDRY_USER_ROLE_ID} --scope '{project_id}'",
    shell=True, capture_output=True, text=True
)
if r.returncode == 0:
    print("  granted: Foundry User on the project")
elif "already exist" in r.stderr.lower():
    print("  already granted: Foundry User on the project")
else:
    raise RuntimeError(r.stderr.strip()[:300])

# Step 7 starts a sandbox that calls the model with this identity, and new role
# assignments take a minute or two to take effect.
print("\nWaiting 90s for RBAC propagation...")
time.sleep(90)
print("Done.")

Agent identity: 00000000-0000-0000-0000-000000000000


  granted: Foundry User on the project

Waiting 90s for RBAC propagation...


Done.


## Step 7: Test the agent

`get_openai_client(agent_name=...)` returns an OpenAI client bound to the agent endpoint
`{project_endpoint}/agents/{agent_name}/endpoint/protocols/openai`. The first call starts a
sandbox: the service provisions compute, starts `python main.py` and waits for its readiness
probe, which takes tens of seconds. The response carries the `agent_session_id` the platform
created; 08-12-02 does more with it.

In [7]:
import openai

agent_client = client.get_openai_client(agent_name=AGENT_NAME)

def ask(question: str, session_id: str | None = None, retries: int = 6):
    """Send one Responses call to the agent endpoint, optionally inside an existing session."""
    extra = {"agent_session_id": session_id} if session_id else {}
    for attempt in range(1, retries + 1):
        try:
            response = agent_client.responses.create(input=question, extra_body=extra)
            break
        except openai.APIStatusError as e:
            # 424 session_not_ready and 5xx while the sandbox starts or RBAC settles
            if e.status_code not in (424, 500, 502, 503) or attempt == retries:
                raise
            print(f"   HTTP {e.status_code} (attempt {attempt}/{retries}), retrying in 20s...")
            time.sleep(20)
    if response.status != "completed":
        raise RuntimeError(f"response {response.status}: {response.error}")
    print(f"\U0001f9d1 {question}\n\U0001f916 {response.output_text}")
    print(f"   agent_session_id={response.model_extra.get('agent_session_id')}")
    return response

t0 = time.time()
ask("Where are you running, and are there any notes saved yet?")
print(f"\nfirst call, including the sandbox cold start: {time.time() - t0:.0f}s")

🧑 Where are you running, and are there any notes saved yet?
🤖 I’m running in session `208d968a95029ed723fbf41067dfe92502cf849ef22c7ef7181ae38e4f262d3` on `adc-sandbox` (`/home/session`).

No notes are saved yet.
   agent_session_id=208d968a95029ed723fbf41067dfe92502cf849ef22c7ef7181ae38e4f262d3

first call, including the sandbox cold start: 18s


## Step 8: The same zip and definition return the same version

Versioning is content-addressable: a new version is minted only when the zip's SHA-256 or the
definition changes. Re-uploading the identical zip with the identical definition returns the
version that already exists, so a redeploy pipeline can post on every run without creating
versions. This is why Step 3 built the zip with fixed timestamps.

In [8]:
again = register(REMOTE_ZIP, "remote_build", "Notes agent; dependencies installed by the service from requirements.txt")
print(f"re-upload returned v{again.version}; the first upload created v{remote.version}")
assert again.version == remote.version, "the same zip and definition should not mint a new version"

re-upload returned v1; the first upload created v1


## Step 9: Download the deployed zip and check its hash

`download_code` returns the zip the service holds for a version. Comparing its SHA-256 with the
one you uploaded, and with the version's `content_hash`, shows exactly what is deployed. An
image-based agent such as the 08-03 one returns `409 AgentNotCodeBased` here.

In [9]:
downloaded = b"".join(client.agents.download_code(agent_name=AGENT_NAME, agent_version=remote.version))
downloaded_sha = hashlib.sha256(downloaded).hexdigest()
print(f"downloaded {len(downloaded):,} bytes")
print(f"sha256 {downloaded_sha}")
print(f"matches the upload:        {downloaded_sha == REMOTE_SHA}")
print(f"matches content_hash:      {downloaded_sha == remote.definition.code_configuration.content_hash}")
print(f"entries: {zipfile.ZipFile(io.BytesIO(downloaded)).namelist()}")

downloaded 4,300 bytes
sha256 526edceedc2bec346b058ee950ff082541920bf6278bd6a05771644898e64a63
matches the upload:        True
matches content_hash:      True
entries: ['main.py', 'requirements.txt']


## Step 10: Bundle the dependencies instead

With `dependency_resolution="bundled"` the service runs the zip as-is and installs nothing, so the
zip must carry the dependencies as extracted modules in a `packages/` folder (not wheel files).
Use it when the build must be reproducible, when the packages are private or wheels-only, or when
the remote build cannot resolve them. The packages are installed for the sandbox, not for this
machine: `manylinux2014` on x86_64 and the Python of the runtime, binary wheels only.

In [10]:
PACKAGES_DIR   = BUILD_DIR / "packages"
PYTHON_VERSION = RUNTIME.removeprefix("python_").replace("_", ".")   # python_3_13 -> 3.13

r = subprocess.run(
    f"uv pip install -q -r {AGENT_DIR / 'requirements.txt'} --target {PACKAGES_DIR} "
    f"--python-platform x86_64-manylinux2014 --python-version {PYTHON_VERSION} --only-binary :all:",
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(r.stderr[-2000:])
native = sorted(p.name for p in PACKAGES_DIR.rglob("*.so"))
print(f"{sum(1 for p in PACKAGES_DIR.iterdir() if p.is_dir() and not p.name.endswith('.dist-info'))} packages, "
      f"{len(native)} compiled extensions, e.g. {native[0]}")

files = {"main.py": AGENT_DIR / "main.py", "requirements.txt": AGENT_DIR / "requirements.txt"}
files.update({p.relative_to(BUILD_DIR).as_posix(): p for p in PACKAGES_DIR.rglob("*") if p.is_file()})
BUNDLED_ZIP = BUILD_DIR / "bundled.zip"
BUNDLED_SHA = build_zip(BUNDLED_ZIP, files)
print(f"{BUNDLED_ZIP}: {BUNDLED_ZIP.stat().st_size / 1e6:.1f} MB, {len(files):,} entries (limit 250 MB)")
print(f"sha256 {BUNDLED_SHA}")

75 packages, 22 compiled extensions, e.g. _cffi_backend.cpython-313-x86_64-linux-gnu.so


build/bundled.zip: 24.0 MB, 5,602 entries (limit 250 MB)
sha256 73129df4397be879b7953b48a051be0e64dd1665d5f224d91dd2a883b652be9c


In [11]:
bundled = register(BUNDLED_ZIP, "bundled", "Notes agent; dependencies bundled in the zip")
print(f"Agent registered: {bundled.name} v{bundled.version}, status={label(bundled.status)}")
print(f"content_hash matches the zip: {bundled.definition.code_configuration.content_hash == BUNDLED_SHA}\n")

BUNDLED_SECONDS = wait_active(bundled.version)
print(f"\nTime to active   remote_build: {REMOTE_BUILD_SECONDS:5.0f}s   bundled: {BUNDLED_SECONDS:5.0f}s")

v = client.agents.get_version(agent_name=AGENT_NAME, agent_version=bundled.version)
print(f"same agent identity as v{remote.version}: {v.instance_identity.principal_id == AGENT_PRINCIPAL_ID}")

Agent registered: contoso-rm-notes-agent v2, status=active
content_hash matches the zip: True



      0s  active

Time to active   remote_build:    51s   bundled:     0s
same agent identity as v1: True


The agent endpoint now routes to the latest version, so the next call starts a sandbox from the
bundled zip.

In [12]:
t0 = time.time()
ask("Save a note titled 'Deploy check' with the body 'bundled build answers', then list my notes.")
print(f"\nfirst call on v{bundled.version}, including the sandbox cold start: {time.time() - t0:.0f}s")

🧑 Save a note titled 'Deploy check' with the body 'bundled build answers', then list my notes.
🤖 Saved: `notes/deploy-check.md`

Your notes:
- `notes/deploy-check.md`: Deploy check
   agent_session_id=b2c51cfbf132adb5ae902c24ccd8579fc74fe875567cf1c72536f3ad5089563

first call on v2, including the sandbox cold start: 15s


## Step 11: What idle timeouts does the service accept?

Learn says 120 to 3600 seconds. The azd agents extension release notes mention a higher limit
of 240 minutes. The service answers the question itself: an out-of-range value is rejected
before any version is created, and the error message quotes the range it enforces.

In [13]:
from azure.core.exceptions import HttpResponseError

for seconds in (60, 14400):
    try:
        with REMOTE_ZIP.open("rb") as code:
            client.agents.create_version_from_code(
                agent_name=AGENT_NAME,
                definition=definition("remote_build", idle_timeout_seconds=seconds),
                code=code,
            )
        print(f"{seconds:>6}s: accepted")
    except HttpResponseError as e:
        print(f"{seconds:>6}s: HTTP {e.status_code} {str(e).splitlines()[0]}")

    60s: HTTP 400 (bad_request) 'Session Configuration Idle Timeout Seconds' must be between 00:02:00 and 01:00:00. You entered 00:01:00. [Request ID: 1931c2f2e855e52d6d4ebb14a5ec1931]
 14400s: HTTP 400 (bad_request) 'Session Configuration Idle Timeout Seconds' must be between 00:02:00 and 01:00:00. You entered 04:00:00. [Request ID: 945f04c00de55639b08ce037bf0c3823]


## Cleanup

08-12-02 needs the agent, so leave it in place. To remove it later, uncomment the lines below.
`force=True` also deletes the agent's sessions; without it, the delete fails with 409 while
sessions exist.

In [14]:
# Uncomment to delete the agent and all its versions.
#
# result = client.agents.delete(agent_name=AGENT_NAME, force=True)
# print(f"Deleted {AGENT_NAME}: {result.deleted}")